# SageMaker Pipeline — process -> train, run locally

A `LocalPipelineSession` pipeline: a **ProcessingStep** (`prep.py`) generates the credit data and a **TrainingStep** fits the CatBoost model on it — wired together as on AWS, with inter-step artifacts passing through oblako's **S3Proxy**. Mirrors `batch_scoring/local/training/sagemaker_pipeline.py`.

Prereqs: `oblako up s3proxy`, Docker, `pip install 'oblako[sagemaker,notebook]'`.

## Wire S3Proxy as the artifact store + build the container

In [ ]:
import os, tempfile
os.environ.setdefault("AWS_DEFAULT_REGION", "us-east-1")
os.environ.setdefault("AWS_ACCESS_KEY_ID", "test")
os.environ.setdefault("AWS_SECRET_ACCESS_KEY", "test")
# SageMaker passes step artifacts via S3 -> point it at S3Proxy (path-style, checksum-off)
cfg = os.path.join(tempfile.mkdtemp(), "aws-config")
open(cfg, "w").write("[default]\nregion = us-east-1\nrequest_checksum_calculation = when_required\n"
                     "s3 =\n    addressing_style = path\n")
os.environ["AWS_ENDPOINT_URL_S3"] = "http://localhost:9000"
os.environ["AWS_REQUEST_CHECKSUM_CALCULATION"] = "when_required"
os.environ["AWS_CONFIG_FILE"] = cfg

from oblako.services import S3ProxyService, SageMakerService
from sagemaker.estimator import Estimator
from sagemaker.processing import ProcessingOutput, ScriptProcessor
from sagemaker.workflow.pipeline import Pipeline
from sagemaker.workflow.pipeline_context import LocalPipelineSession
from sagemaker.workflow.steps import ProcessingStep, TrainingStep

IMAGE, ROLE, BUCKET = "oblako-credit-risk:latest", "arn:aws:iam::000000000000:role/dummy", "credit-risk-pipeline"
SageMakerService().build_image(path="container", tag=IMAGE)
s3 = S3ProxyService().get_client()
try:
    s3.create_bucket(Bucket=BUCKET)
except Exception:
    pass

## Define and run the pipeline

In [ ]:
session = LocalPipelineSession()
session.default_bucket = lambda: BUCKET

processor = ScriptProcessor(image_uri="python:3.11-slim", command=["python"],
                            instance_type="local", instance_count=1, role=ROLE,
                            sagemaker_session=session)
prep = ProcessingStep(name="GenerateData", processor=processor, code="prep.py",
                      outputs=[ProcessingOutput(output_name="train", source="/opt/ml/processing/output")])

estimator = Estimator(image_uri=IMAGE, role=ROLE, instance_count=1, instance_type="local",
                      sagemaker_session=session,
                      hyperparameters={"iterations": "150", "depth": "6", "learning-rate": "0.1",
                                       "factor": "20", "offset": "600"})
train_step = TrainingStep(name="TrainCreditModel", estimator=estimator,
    inputs={"train": prep.properties.ProcessingOutputConfig.Outputs["train"].S3Output.S3Uri})

pipeline = Pipeline(name="oblako-credit-risk", steps=[prep, train_step], sagemaker_session=session)
pipeline.upsert(role_arn=ROLE)
execution = pipeline.start()
print("pipeline:", execution.status)
for step in execution.list_steps()["PipelineExecutionSteps"]:
    print(" ", step["StepName"], step["StepStatus"])